<a href="https://colab.research.google.com/github/evapatel123/EVio/blob/main/Copy_of_Untitled5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q gradio huggingface_hub sentence-transformers torch

In [3]:
%%writefile sidebar.py
import gradio as gr

class SidebarMenu(gr.HTML):
    def __init__(
        self, menu_data, value=None, open=True, position="right", width=300, **kwargs
    ):

        html_template = """
        <div class="sidebar ${position} ${open ? 'open' : ''}" style="width: ${width}px; ${position}: -${width}px;">
            <button class="toggle-button" aria-label="Toggle Sidebar">
                <i data-lucide="chevron-right" class="toggle-icon"></i>
            </button>

            <div class="sidebar-content">
                <div class="sidebar-wrapper">
                    ${menu_data.map(item => {
                        if (item.type === 'group') {
                            return `
                            <div class="menu-group">
                                <div class="group-header">
                                    <div class="header-left">
                                        <i data-lucide="${item.icon}" class="icon" style="color: ${item.color || '#a9b1d6'};"></i>
                                        <span class="label">${item.label}</span>
                                    </div>
                                    <i data-lucide="chevron-down" class="arrow icon"></i>
                                </div>
                                <div class="group-children">
                                    ${item.children.map(child => `
                                        <div class="menu-item ${value === child.id ? 'active' : ''}" data-id="${child.id}">
                                            <i data-lucide="${child.icon}" class="icon" style="color: ${child.color || '#a9b1d6'};"></i>
                                            <span class="label">${child.label}</span>
                                        </div>
                                    `).join('')}
                                </div>
                            </div>`;
                        } else {
                            return `
                            <div class="menu-item ${value === item.id ? 'active' : ''}" data-id="${item.id}">
                                <i data-lucide="${item.icon}" class="icon" style="color: ${item.color || '#a9b1d6'};"></i>
                                <span class="label">${item.label}</span>
                            </div>`;
                        }
                    }).join('')}
                </div>
            </div>
        </div>
        """

        css_template = """
            .sidebar {
                display: flex;
                flex-direction: column;
                position: fixed;
                top: 0;
                height: 100%;
                background-color: var(--background-fill-secondary);
                transform: translateX(0%);
                z-index: 1000;
                transition: transform 0.3s ease-in-out;
                border-right: 1px solid var(--border-color-primary);
                color: var(--body-text-color);
                font-family: var(--font-sans-serif);
            }

            .sidebar.open.left {
                transform: translateX(100%);
            }

            .sidebar.open.right {
                transform: translateX(-100%);
            }

            .toggle-button {
                position: absolute;
                top: 20px;
                background: var(--background-fill-secondary);
                border: 1px solid var(--border-color-primary);
                cursor: pointer;
                padding: 8px;
                display: flex;
                align-items: center;
                justify-content: center;
                width: 28px;
                height: 32px;
                z-index: 1001;
                transition: all 0.3s ease;
                border-radius: 0;
            }
            .toggle-button * {
                pointer-events: none;
            }
            .toggle-button:hover {
                background: var(--background-fill-primary);
                border-color: var(--color-accent);
            }

            .toggle-icon {
                width: 20px;
                height: 20px;
                color: var(--body-text-color-subdued);
                transition: transform 0.3s ease-in-out;
                stroke-width: 2.5;
            }

            .sidebar.open .toggle-icon {
                transform: rotate(180deg);
            }

            .sidebar.left .toggle-button {
                left: 100%;
                border-radius: 0 8px 8px 0;
            }

            .sidebar.right .toggle-button {
                right: 100%;
                border-radius: 8px 0 0 8px;
            }

            .sidebar-content {
                height: 100%;
                overflow-y: auto;
                overflow-x: hidden;
            }

            .sidebar-content::-webkit-scrollbar {
                width: 6px;
            }

            .sidebar-content::-webkit-scrollbar-thumb {
                background: var(--border-color-primary);
                border-radius: 10px;
            }

            .sidebar-wrapper {
                font-family: 'Inter', sans-serif;
                padding: 20px 15px;
            }

            .menu-item {
                display: flex;
                align-items: center;
                padding: 14px 18px;
                margin-bottom: 6px;
                border-radius: 12px;
                cursor: pointer;
                font-size: 15px;
                font-weight: 500;
                transition: all 0.3s ease;
            }

            .menu-item:hover {
                background: linear-gradient(90deg,#6366F1,#8B5CF6);
                color: white;
                transform: translateX(8px);
                box-shadow: 0 6px 18px rgba(99,102,241,.35);
            }

            .menu-item:hover .icon {
                transform: scale(1.15);
            }

            .menu-item.active {
                background: linear-gradient(90deg,#6366F1,#8B5CF6);
                color: white !important;
                box-shadow: 0 6px 18px rgba(99,102,241,.4);
            }

            .group-header {
                display: flex;
                justify-content: space-between;
                align-items: center;
                padding: 12px 15px;
                margin-bottom: 4px;
                border-radius: 8px;
                cursor: pointer;
                font-size: 14px;
                font-weight: 600;
                color: var(--body-text-color);
                transition: background-color 0.2s;
            }

            .group-header:hover {
                background: linear-gradient(90deg,#6366F1,#8B5CF6);
                color: white;
                transform: translateX(8px);
            }

            .header-left {
                display: flex;
                align-items: center;
            }

            .group-children {
                display: none;
                padding-left: 15px;
                margin-top: 4px;
            }

            .icon {
                margin-right: 12px;
                width: 18px;
                height: 18px;
                stroke: currentColor;
                stroke-width: 2.2;
                color: inherit;
                transition: transform 0.3s ease;
            }

            .arrow {
                width: 14px;
                height: 14px;
                transition: transform 0.3s ease;
                color: var(--body-text-color-subdued);
                margin-right: 0;
            }

            @media (max-width: 768px) {
                .sidebar {
                    width: 100vw !important;
                }

                .sidebar.left {
                    left: -100vw !important;
                }

                .sidebar.right {
                    right: -100vw !important;
                }
            }
        """

        js_on_load = """
            let openFolders = new Set();

            function initLucide(retries = 8, delay = 200) {
                if (window.lucide) {
                    window.lucide.createIcons();
                    element.querySelectorAll('i[data-lucide]').forEach(i => {
                        const svg = i.querySelector('svg');
                        if (svg) {
                            let desiredColor = i.style.color || getComputedStyle(i).color;
                            if (desiredColor &&
                                desiredColor !== 'rgb(0,0,0)' && desiredColor !== '#000000' &&
                                desiredColor !== 'rgb(255,255,255)' && desiredColor !== '#ffffff') {
                                svg.style.color = desiredColor;
                                const shapes = svg.querySelectorAll('path, line, polyline, circle, rect, polygon');
                                shapes.forEach(shape => {
                                    shape.setAttribute('stroke', desiredColor);
                                    shape.style.stroke = desiredColor + ' !important';
                                });
                                if (svg.hasAttribute('fill') && svg.getAttribute('fill') === 'currentColor') {
                                    svg.setAttribute('fill', desiredColor);
                                }
                                svg.querySelectorAll('[fill="currentColor"]').forEach(el => {
                                    el.setAttribute('fill', desiredColor);
                                });
                            }
                        }
                    });
                } else if (retries > 0) {
                    setTimeout(() => initLucide(retries - 1, delay), delay);
                }
            }

            function applyFolderStates() {
                element.querySelectorAll('.group-header').forEach(header => {
                    const label = header.querySelector('.label')?.innerText.trim();
                    if (!label) return;
                    const children = header.nextElementSibling;
                    const arrow = header.querySelector('.arrow');
                    if (children && arrow) {
                        const isOpen = openFolders.has(label);
                        children.style.display = isOpen ? 'block' : 'none';
                        arrow.style.transform = isOpen ? 'rotate(180deg)' : 'rotate(0deg)';
                    }
                });
            }

            initLucide();
            applyFolderStates();

            const observer = new MutationObserver(() => {
                setTimeout(() => {
                    applyFolderStates();
                    initLucide();
                }, 50);
            });

            observer.observe(element, {
                childList: true,
                subtree: true,
                attributes: true,
                characterData: true
            });

            element.addEventListener('click', (e) => {
                if (e.target.closest('.toggle-button')) {
                    const sidebar = element.querySelector('.sidebar');
                    sidebar.classList.toggle('open');
                    trigger(sidebar.classList.contains('open') ? 'expand' : 'collapse');
                    return;
                }

                const header = e.target.closest('.group-header');
                if (header) {
                    const label = header.querySelector('.label')?.innerText.trim();
                    if (label) {
                        if (openFolders.has(label)) openFolders.delete(label);
                        else openFolders.add(label);
                    }
                    applyFolderStates();
                    return;
                }

                const item = e.target.closest('.menu-item');
                if (item) {
                    props.value = item.dataset.id;
                    trigger('change');

                    if (window.innerWidth <= 768) {
                        element.querySelector('.sidebar')?.classList.remove('open');
                    }

                    setTimeout(() => {
                        applyFolderStates();
                        initLucide();
                    }, 0);
                }
            });
        """

        super().__init__(
            value=value,
            html=html_template,
            **kwargs
        )
        self.css = css_template
        self.js = js_on_load

    def api_info(self):
        return {"type": "string"}

Writing sidebar.py


In [ ]:
# STEP 1: INSTALL REQUIRED PACKAGES
# !pip install -q gradio transformers accelerate sentence-transformers torch

import os
from threading import Thread
import gradio as gr
from sentence_transformers import SentenceTransformer
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TextIteratorStreamer

# STEP 2: LOAD KNOWLEDGE BASE WITH FALLBACK
knowledge_file_path = "knowledge.txt"

if os.path.exists(knowledge_file_path):
    print("✨ Found 'knowledge.txt'! Processing knowledge chunks...")
    with open(knowledge_file_path, "r", encoding="utf-8") as file:
        recent = file.read()
else:
    print(
        "⚠️ 'knowledge.txt' not found. Creating automatic placeholder context"
        " data..."
    )
    recent = (
        "Pack essentials like twin XL bedding, surge protectors, and a shower"
        " caddy.\nManage your time with a planner or digital calendar.\nHandle"
        " roommate conflicts by setting boundaries early and communicating"
        " openly."
    )

cleaned_text = recent.strip()
chunks = cleaned_text.split("\n")
cleaned_chunks = [chunk.strip() for chunk in chunks if chunk.strip()]


# STEP 3: SETUP LOCAL ENVIRONMENT & RAG EMBEDDINGS
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Running sentence-transformers on: {device.upper()}")

embedding_model = SentenceTransformer("all-MiniLM-L6-v2", device=device)
chunk_embeddings = embedding_model.encode(
    cleaned_chunks, convert_to_tensor=True
)


def get_top_chunks(query):
    query_str = str(query)
    query_embedding = embedding_model.encode(query_str, convert_to_tensor=True)
    query_embedding_normalized = query_embedding / query_embedding.norm()
    chunk_embeddings_normalized = chunk_embeddings / chunk_embeddings.norm(
        dim=1, keepdim=True
    )

    similarities = torch.matmul(
        chunk_embeddings_normalized, query_embedding_normalized
    )
    top_indices = torch.topk(
        similarities, k=min(3, len(cleaned_chunks))
    ).indices

    top_chunks = [cleaned_chunks[i] for i in top_indices]
    return top_chunks


# STEP 4: LOAD THE MAIN LLM LOCALLY ON T4 GPU
print("🔄 Loading Qwen2.5 locally onto your Colab GPU... (Takes ~1-2 minutes)")

model_id = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, torch_dtype="auto", device_map="auto"
)

print("✨ Local LLM Loaded! Building your exact original EVio UI Layout...")


# STEP 5: ORIGINAL FIREFLY CSS CUSTOM PALETTE
custom_css = """
:root, .gradio-container {
    --background-fill-primary: #f2f7f2 !important;
    --background-fill-secondary: #e6eee5 !important;
    --block-background-fill: #ffffff !important;

    --primary-50: #f2f7f2 !important;
    --primary-100: #e6eee5 !important;
    --primary-200: #cbdcc9 !important;
    --primary-500: #556b54 !important;
    --primary-600: #425441 !important;

    --body-text-color: #384537 !important;
    --block-label-text-color: #556b54 !important;
    --input-placeholder-color: #8fa38e !important;

    --radius-md: 16px !important;
    --radius-lg: 24px !important;
}

button.primary-btn {
    background: linear-gradient(135deg, #556b54, #425441) !important;
    color: white !important;
    border: none !important;
    padding: 10px 20px !important;
    border-radius: 8px !important;
    cursor: pointer !important;
}
"""

# STEP 6: BULLETPROOF STABLE STATE PIPELINE
def add_user_message(message, chat_history):
    if not message.strip():
        return "", chat_history

    chat_history.append(gr.ChatMessage(role="user", content=message))
    return "", chat_history


def generate_bot_response(chat_history):
    if not chat_history:
        yield chat_history
        return

    last_turn = chat_history[-1]
    if hasattr(last_turn, "content"):
        user_message = str(last_turn.content)
    elif isinstance(last_turn, dict):
        user_message = str(last_turn.get("content", ""))
    elif isinstance(last_turn, (list, tuple)):
        user_message = str(last_turn[0])
    else:
        user_message = str(last_turn)

    college_context = get_top_chunks(user_message)

    system_prompt = (
        "You are EVio, the firefly AI college transition assistant. Your goal is"
        " to help high school students smoothly transition to college. Provide"
        " actionable advice, step-by-step guidance, and encouraging support."
        f" Use the following database context to answer questions: {college_context}"
    )

    messages = [{"role": "system", "content": system_prompt}]
    for turn in chat_history:
        role = (
            turn.role
            if hasattr(turn, "role")
            else turn.get("role")
            if isinstance(turn, dict)
            else "user"
        )
        content = (
            turn.content
            if hasattr(turn, "content")
            else turn.get("content", "")
            if isinstance(turn, dict)
            else str(turn)
        )
        if content:
            messages.append({"role": role, "content": str(content)})

    if len(messages) > 12:
        messages = [messages[0]] + messages[-10:]

    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    streamer = TextIteratorStreamer(
        tokenizer, skip_prompt=True, skip_special_tokens=True
    )
    generation_kwargs = dict(
        model_inputs, streamer=streamer, max_new_tokens=512, temperature=0.8
    )

    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    chat_history.append(gr.ChatMessage(role="assistant", content=""))

    partial_text = ""
    for new_text in streamer:
        partial_text += new_text
        chat_history[-1] = gr.ChatMessage(role="assistant", content=partial_text)
        yield chat_history


def respond_to_example(example_text, chat_history):
    chat_history.append(gr.ChatMessage(role="user", content=example_text))
    for updated_history in generate_bot_response(chat_history):
        yield updated_history


# STEP 7: VISUAL LAYOUT BLOCKS BUILD
with gr.Blocks(
    theme=gr.Theme.from_hub("allenai/gradio-theme")
) as demo:
    with gr.Sidebar():
        gr.Markdown("## Navigation")

        home_btn = gr.Button("EVio Chatbot", variant="primary")
        quiz_btn = gr.Button("Majors Quiz", variant="secondary")
        dashboard_btn = gr.Button("Dashboard", variant="secondary")
        scholarships_btn = gr.Button("Scholarships", variant="secondary")
        planner_btn = gr.Button("Academic Planner", variant="secondary")
        settings_btn = gr.Button("Settings", variant="secondary")

    # --- SECTION 1: EVIO CHATBOT (HOME) ---
    with gr.Group(visible=True) as chatbot_section:
        if os.path.exists("evio-banner.png"):
            gr.Image(
                value="evio-banner.png", show_label=False, elem_id="top-image"
            )

        gr.Markdown("## 🎓 Meet EVio – Your College Transition Companion!")
        gr.Markdown(
            "**EVio, the firefly**, helps high school students successfully"
            " navigate the leap to higher education. Ask about packing lists,"
            " managing financial aid, adjusting to dorm life, or mastering"
            " college academics! 🌟 *Your AI guide. Your learning journey.*"
        )

        chatbot = gr.Chatbot(label="EVio Chat Assistant")

        with gr.Row():
            msg_input = gr.Textbox(
                show_label=False,
                placeholder=(
                    "Type your question about transitioning to college here..."
                ),
                scale=8,
            )
            submit_btn = gr.Button(
                "Send",
                variant="primary",
                scale=1,
                elem_classes=["primary-btn"],
            )

        msg_input.submit(
            fn=add_user_message,
            inputs=[msg_input, chatbot],
            outputs=[msg_input, chatbot],
            queue=True,
        ).then(
            fn=generate_bot_response, inputs=[chatbot], outputs=[chatbot]
        )

        submit_btn.click(
            fn=add_user_message,
            inputs=[msg_input, chatbot],
            outputs=[msg_input, chatbot],
            queue=True,
        ).then(
            fn=generate_bot_response, inputs=[chatbot], outputs=[chatbot]
        )

        gr.Markdown("### Try asking EVio:")
        examples = [
            "What should I pack for my dorm room?",
            "How do I manage my time between classes and studying?",
            "Tips for handling college roommate conflicts?",
        ]

        with gr.Row():
            for example_text in examples:
                ex_btn = gr.Button(example_text, variant="secondary")
                ex_btn.click(
                    fn=respond_to_example,
                    inputs=[gr.State(example_text), chatbot],
                    outputs=[chatbot],
                )

    # --- SECTION 2: MAJORS QUIZ ---
    with gr.Group(visible=False) as quiz_section:
        gr.Markdown("## 📝 Majors Quiz")
        gr.Markdown(
            "Discover academic majors that match your strengths and interests."
        )

    # --- SECTION 3: DASHBOARD ---
    with gr.Group(visible=False) as dashboard_section:
        gr.Markdown("## 📊 Dashboard")
        gr.Markdown("Track your college application and transition progress.")

    # --- SECTION 4: SCHOLARSHIPS ---
    with gr.Group(visible=False) as scholarships_section:
        gr.Markdown("## 💰 Scholarships")
        gr.Markdown("Explore aid options and active scholarship listings.")

    # --- SECTION 5: ACADEMIC PLANNER ---
    with gr.Group(visible=False) as planner_section:
        gr.Markdown("## 📅 Academic Planner")
        gr.Markdown("Schedule your semester and stay on top of deadlines.")

    # --- SECTION 6: SETTINGS ---
    with gr.Group(visible=False) as settings_section:
        gr.Markdown("## ⚙️ Settings")
        gr.Markdown("Manage your account and app preferences.")

    # NAVIGATION LOGIC FOR SIDEBAR BUTTONS
    def go_chatbot():
        return (
            gr.update(variant="primary"),
            gr.update(variant="secondary"),
            gr.update(variant="secondary"),
            gr.update(variant="secondary"),
            gr.update(variant="secondary"),
            gr.update(variant="secondary"),
            gr.update(visible=True),
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(visible=False),
        )

    def go_quiz():
        return (
            gr.update(variant="secondary"),
            gr.update(variant="primary"),
            gr.update(variant="secondary"),
            gr.update(variant="secondary"),
            gr.update(variant="secondary"),
            gr.update(variant="secondary"),
            gr.update(visible=False),
            gr.update(visible=True),
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(visible=False),
        )

    def go_dashboard():
        return (
            gr.update(variant="secondary"),
            gr.update(variant="secondary"),
            gr.update(variant="primary"),
            gr.update(variant="secondary"),
            gr.update(variant="secondary"),
            gr.update(variant="secondary"),
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(visible=True),
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(visible=False),
        )

    def go_scholarships():
        return (
            gr.update(variant="secondary"),
            gr.update(variant="secondary"),
            gr.update(variant="secondary"),
            gr.update(variant="primary"),
            gr.update(variant="secondary"),
            gr.update(variant="secondary"),
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(visible=True),
            gr.update(visible=False),
            gr.update(visible=False),
        )

    def go_planner():
        return (
            gr.update(variant="secondary"),
            gr.update(variant="secondary"),
            gr.update(variant="secondary"),
            gr.update(variant="secondary"),
            gr.update(variant="primary"),
            gr.update(variant="secondary"),
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(visible=True),
            gr.update(visible=False),
        )

    def go_settings():
        return (
            gr.update(variant="secondary"),
            gr.update(variant="secondary"),
            gr.update(variant="secondary"),
            gr.update(variant="secondary"),
            gr.update(variant="secondary"),
            gr.update(variant="primary"),
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(visible=True),
        )

    # WIRE CLICK EVENTS
    all_btns_and_sections = [
        home_btn,
        quiz_btn,
        dashboard_btn,
        scholarships_btn,
        planner_btn,
        settings_btn,
        chatbot_section,
        quiz_section,
        dashboard_section,
        scholarships_section,
        planner_section,
        settings_section,
    ]

    home_btn.click(fn=go_chatbot, outputs=all_btns_and_sections)
    quiz_btn.click(fn=go_quiz, outputs=all_btns_and_sections)
    dashboard_btn.click(fn=go_dashboard, outputs=all_btns_and_sections)
    scholarships_btn.click(fn=go_scholarships, outputs=all_btns_and_sections)
    planner_btn.click(fn=go_planner, outputs=all_btns_and_sections)
    settings_btn.click(fn=go_settings, outputs=all_btns_and_sections)

# STEP 8: LAUNCH SERVER RUNTIME
if __name__ == "__main__":
    demo.launch(debug=True, share=True, inline=False)

✨ Found 'knowledge.txt'! Processing knowledge chunks...
🚀 Running sentence-transformers on: CUDA


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

🔄 Loading Qwen2.5 locally onto your Colab GPU... (Takes ~1-2 minutes)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

✨ Local LLM Loaded! Building your exact original EVio UI Layout...


/tmp/ipykernel_554/838964553.py:193: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://cc992f79b9dfba5220.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
